In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

## Preparing the dataset

Preparation:

- Fill missing values with zeros.
- Do the split.
- Use DictVectorizer(sparse=True) to turn the dataframes into matrices.

In [2]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [4]:
df = df.fillna(0.0)

In [5]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [7]:
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [8]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

In [9]:
X_train_df = df_train.drop(columns='fuel_efficiency_mpg')
X_val_df = df_val.drop(columns='fuel_efficiency_mpg')
X_test_df = df_test.drop(columns='fuel_efficiency_mpg')

In [10]:
train_dicts = X_train_df.to_dict(orient='records')
val_dicts = X_val_df.to_dict(orient='records')
test_dicts = X_test_df.to_dict(orient='records')

In [11]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)

X_val = dv.transform(val_dicts)

X_test = dv.transform(test_dicts)

In [12]:
X_train.shape, X_val.shape, X_test.shape

((6000, 16), (2000, 16), (2000, 16))

## Question 1  

Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.

Train a model with max_depth=1.

Which original feature is used for splitting the data? 

In [13]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1)

dt.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [14]:
from sklearn.tree import export_text

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



Feature 'model_year' is used for splitting the data.



## Question 2  

Train a random forest regressor with these parameters:
- n_estimators=10
- random_state=1
- n_jobs=-1 (optional - to make training faster)

What's the RMSE of this model on the validation data?

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(rmse)

1.837054054730018


RMSE of this model on the validation data is 1.837.

## Question 3  

Now let's experiment with the n_estimators parameter

- Try these values of this parameter: [10, 50, 100, 150].
- Set random_state to 1.
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

In [16]:
for n in [10, 50, 100, 150]:

    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
  
    rf.fit(X_train, y_train)


    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(n, round(rmse, 3))

10 1.837
50 1.772
100 1.768
150 1.769


The lowest validation RMSE is 1.768, achieved with n_estimators = 100.

## Question 4 

Let's select the best max_depth:

Try different values of max_depth: [10, 15, 20, 25]

For each of these values,
- try different values of n_estimators from [10, 50, 100, 150]
- calculate the mean RMSE
- Fix the random seed: random_state=1

What's the best max_depth, using the mean RMSE?

In [17]:
for depth in [10, 15, 20, 25]:
    scores = []

    for n in [10, 50, 100, 150]:

        rf = RandomForestRegressor(
            n_estimators=n,
            random_state=1,
            n_jobs=-1,
            max_depth=depth
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        scores.append(rmse)

    print(depth, np.mean(scores))

10 1.7554047863550186
15 1.7839526938716295
20 1.7882420685694522
25 1.786317363879773


The best max_depth, using the mean RMSE, is 10 because it gives the lowest mean RMSE.

## Question 5 
Train the model with these parameters:
- n_estimators=10,
- max_depth=20,
- random_state=1,
- n_jobs=-1 (optional)

Get the feature importance information from this model
What's the most important feature (among these 4)?

In [18]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)


,n_estimators,10
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [19]:
for feature, importance in zip(
    dv.get_feature_names_out(),
    rf.feature_importances_
):
    print(feature, round(importance, 3))

acceleration 0.055
drivetrain=All-wheel drive 0.008
drivetrain=Front-wheel drive 0.02
drivetrain=Rear-wheel drive 0.006
engine_displacement 0.061
fuel_type=Diesel 0.017
fuel_type=Gasoline 0.161
fuel_type=Hybrid 0.016
horsepower 0.078
model_year 0.296
num_cylinders 0.007
num_doors 0.019
origin=Asia 0.006
origin=Europe 0.004
origin=USA 0.042
vehicle_weight 0.204


The most important feature (among these 4) is vehicle_weight.

## Question 6  

Now let's train an XGBoost model! For this question, we'll tune the eta parameter:
- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds.

Now change eta from 0.3 to 0.1.

Which eta leads to the best RMSE score on the validation dataset?

In [20]:
%pip install xgboost


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import xgboost as xgb

print(xgb.__version__)

3.2.0


In [22]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [
    (dtrain, 'train'),
    (dval, 'val')
]

scores = {}

In [23]:
%%capture output

xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    verbose_eval=5,
    evals=watchlist
)


In [24]:
print(output.stdout)

[0]	train-rmse:2.40653	val-rmse:2.46782
[5]	train-rmse:1.59328	val-rmse:1.76216
[10]	train-rmse:1.47125	val-rmse:1.73205
[15]	train-rmse:1.41908	val-rmse:1.73512
[20]	train-rmse:1.39368	val-rmse:1.73766
[25]	train-rmse:1.35837	val-rmse:1.74359
[30]	train-rmse:1.33243	val-rmse:1.74548
[35]	train-rmse:1.29290	val-rmse:1.75198
[40]	train-rmse:1.25326	val-rmse:1.76296
[45]	train-rmse:1.21527	val-rmse:1.77244
[50]	train-rmse:1.18645	val-rmse:1.77517
[55]	train-rmse:1.14797	val-rmse:1.78668
[60]	train-rmse:1.10265	val-rmse:1.79401
[65]	train-rmse:1.06590	val-rmse:1.80143
[70]	train-rmse:1.04604	val-rmse:1.80444
[75]	train-rmse:1.02727	val-rmse:1.80682
[80]	train-rmse:0.99487	val-rmse:1.81308
[85]	train-rmse:0.96970	val-rmse:1.81600
[90]	train-rmse:0.94986	val-rmse:1.82135
[95]	train-rmse:0.91805	val-rmse:1.82544
[99]	train-rmse:0.90106	val-rmse:1.82646



In [25]:
def parse_xgb_output(output):
    results = []

    for line in output.stdout.strip().split('\n'):
        it_line, train_line, val_line = line.split('\t')

        it = int(it_line.strip('[]'))
        train = float(train_line.split(':')[1])
        val = float(val_line.split(':')[1])

        results.append((it, train, val))

    columns = ['num_iter', 'train_rmse', 'val_rmse']

    return pd.DataFrame(results, columns=columns)

In [26]:
scores['eta=0.3'] = parse_xgb_output(output)

scores['eta=0.3'].head()

,num_iter,train_rmse,val_rmse
0,0,2.40653,2.46782
1,5,1.59328,1.76216
2,10,1.47125,1.73205
3,15,1.41908,1.73512
4,20,1.39368,1.73766


In [27]:
%%capture output

xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    verbose_eval=5,
    evals=watchlist
)


In [28]:
scores['eta=0.1'] = parse_xgb_output(output)

In [29]:
scores['eta=0.3'].tail(1)

,num_iter,train_rmse,val_rmse
20,99,0.90106,1.82646


In [30]:
scores['eta=0.1'].tail(1)

,num_iter,train_rmse,val_rmse
20,99,1.31192,1.72481


In [31]:
print(scores['eta=0.3']['val_rmse'].min())
print(scores['eta=0.1']['val_rmse'].min())

1.73205
1.71287


Eta = 0.1 leads to the best RMSE score on the validation dataset.